In [6]:
# !pip install \
#     feedparser \
#     newspaper3k \
#     beautifulsoup4 \
#     requests \
#     pandas \
#     tqdm \
#     natasha \
#     yake \
#     networkx \
#     lxml \
#     nltk \
#     googlenewsdecoder \
#     "setuptools<81"

In [7]:
# !pip install lxml_html_clean

In [8]:
# !pip install --upgrade pip setuptools wheel

In [9]:
import os
import re
import json
import math
import uuid
import feedparser
import requests
import pandas as pd
import networkx as nx

from tqdm import tqdm
from bs4 import BeautifulSoup
from newspaper import Article
from itertools import combinations
from collections import Counter, defaultdict

from natasha import (
    Segmenter,
    NewsEmbedding,
    NewsNERTagger,
    Doc
)

import yake

In [10]:
# =========================
# CONFIG
# =========================

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
}

# Что собирать (general_science пропускается, если уже есть documents.json)
COLLECT = {
    "naukogrady": True,
    "rosatom": True,
    "international": True,
    "general_science": False,
}

OUTPUT_DIR = "data"
RAW_DIR = os.path.join(OUTPUT_DIR, "raw")
RAW_GENERAL = os.path.join(RAW_DIR, "general_science")
RAW_NAUKOGRADY = os.path.join(RAW_DIR, "naukogrady")
RAW_ROSATOM = os.path.join(RAW_DIR, "rosatom")
RAW_INTL = os.path.join(RAW_DIR, "international")
PROCESSED_DIR = os.path.join(OUTPUT_DIR, "processed")
GRAPH_DIR = os.path.join(OUTPUT_DIR, "graphs")
GRAPH_NAUKOGRADY = os.path.join(GRAPH_DIR, "naukogrady")

for d in [
    RAW_DIR, RAW_GENERAL, RAW_NAUKOGRADY, RAW_ROSATOM, RAW_INTL,
    PROCESSED_DIR, GRAPH_DIR, GRAPH_NAUKOGRADY,
]:
    os.makedirs(d, exist_ok=True)

MIN_TEXT_LEN = 400
REQUEST_TIMEOUT = 25
SLEEP_BETWEEN_REQUESTS = 0.3

# --- Наукограды ---
NAUKOGRAD_CITIES = [
    "долгопрудный", "дубна", "обнинск", "королёв", "королев", "фрязино",
    "пущино", "жуковский", "реутов", "троицк", "черноголовка",
    "сколково", "иннополис", "протвино", "биокомбинат", "красноармейск",
]

DOMAIN_MARKERS = [
    "наукоград", "наукограды", "город науки", "нтп",
    "наукоёмк", "наукоемк", "технопарк", "научно-производствен",
    "исследовател", "научный потенциал", "нии", "российская академия наук",
]

SEED_URLS_NAUKOGRADY = [
    "https://trends.rbc.ru/trends/education/60ca120a9a794708a559c080",
    "https://trends.rbc.ru/trends/education/622f2ea59a79476f34ac3c8f",
    "https://trends.rbc.ru/trends/education/625faa909a794776b07fcc6a",
    "https://trends.rbc.ru/trends/education/6440cd219a7947834e9e39d0",
]

GOOGLE_NEWS_NAUKOGRADY = [
    "наукоград Россия",
    "наукограды России",
    "Долгопрудный наукоград 2026",
    "статус наукограда Долгопрудный",
    "наукоёмкая продукция наукоград",
    "федеральная поддержка наукоградов",
    "Сколково наукоград исследователи",
    "Иннополис наукоград",
    "научно-технологический потенциал наукоград",
    "site:trends.rbc.ru наукоград",
    "site:minobrnauki.gov.ru наукоград",
    "Фальков наукоград",
]

RBC_EDUCATION_PAGES = [
    "https://trends.rbc.ru/trends/education",
    "https://trends.rbc.ru/trends/tag/science",
]

RSS_FILTERED = [
    ("https://nauka.tass.ru/rss/v2.xml", "tass"),
    ("https://rg.ru/xml/index.xml", "rg"),
    ("https://scientificrussia.ru/rss", "scientificrussia"),
]

RSS_MATCH_KEYWORDS = DOMAIN_MARKERS + NAUKOGRAD_CITIES + ["наукоград", "наукограды"]

# --- Росатом (контроль) ---
SEED_URLS_ROSATOM = [
    "https://strana-rosatom.ru/2016/06/14/sergej-kirienko/",
]

GOOGLE_NEWS_ROSATOM = [
    "site:strana-rosatom.ru интервью",
    "site:strana-rosatom.ru Росатом технологии",
    "Росатом оценка рынка",
    "Кириенко Росатом интервью",
]

ROSATOM_KEYWORDS = ["росатом", "атомн", "ядерн", "аэс", "кириенко", "твэл", "росэнергоатом"]

# --- Международное НТС ---
GOOGLE_NEWS_INTL = [
    "международное научно-техническое сотрудничество Россия",
    "технологический суверенитет Россия наука",
    "БРИКС научное сотрудничество",
    "постсанкционная кооперация технологии Россия",
    "site:issek.hse.ru международное научно-техническое",
    "Россия Африка научное сотрудничество",
    "Россия Азия технологическое сотрудничество",
]

INTL_KEYWORDS = [
    "брикс", "суверенитет", "международн", "кооперац", "санкци",
    "сотрудничеств", "технологическ", "экспорт", "импортозамещ",
]

RSS_FEEDS_GENERAL = [
    "https://scientificrussia.ru/rss",
    "https://nauka.tass.ru/rss/v2.xml",
    "https://rg.ru/xml/index.xml",
]

# --- Граф ---
GRAPH_TOP_NODES = 400
GRAPH_MIN_WEIGHT = 2
GRAPH_MIN_PMI = 5.0

# =========================
# NATASHA
# =========================

segmenter = Segmenter()
emb = NewsEmbedding()
ner_tagger = NewsNERTagger(emb)

# =========================
# YAKE
# =========================

kw_extractor = yake.KeywordExtractor(
    lan="ru",
    n=3,
    top=20
)

# =========================
# HELPERS
# =========================

def clean_text(text):

    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\n+", "\n", text)

    return text.strip()


def parse_article(url):

    try:
        article = Article(url, language='ru')

        article.download()
        article.parse()

        text = clean_text(article.text)

        return {
            "title": article.title,
            "text": text,
            "authors": article.authors,
            "top_image": article.top_image
        }

    except Exception as e:

        print(f"[ARTICLE ERROR] {url} -> {e}")

        return None


def extract_entities(text):

    doc = Doc(text)

    doc.segment(segmenter)
    doc.tag_ner(ner_tagger)

    entities = []

    for span in doc.spans:

        entities.append({
            "text": span.text,
            "type": span.type
        })

    return entities


def extract_keywords(text):

    kws = kw_extractor.extract_keywords(text)

    return [x[0] for x in kws]


def split_sentences(text):
    return [s.strip() for s in re.split(r"[.!?]+", text) if len(s.strip()) > 20]


def resolve_url(url):
    if "news.google.com" not in url:
        return url
    try:
        from googlenewsdecoder import gnewsdecoder
        result = gnewsdecoder(url, interval=1)
        if result.get("status") and result.get("decoded_url"):
            return result["decoded_url"]
    except Exception as e:
        print(f"[DECODE ERROR] {url[:80]}... -> {e}")
    return url


def parse_rss(url):
    resp = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
    resp.raise_for_status()
    return feedparser.parse(resp.content)


def text_blob(title, text):
    return f"{title or ''} {text or ''}".lower()


def match_keywords(blob, keywords):
    return [k for k in keywords if k in blob]


def classify_naukograd(title, text):
    blob = text_blob(title, text)
    title_l = (title or "").lower()
    head = blob[:1000]

    if any(k in title_l for k in ["наукоград", "наукограды", "город науки"]):
        return True, "title_keyword", match_keywords(blob, NAUKOGRAD_CITIES)[:1]

    cities = match_keywords(blob, NAUKOGRAD_CITIES)
    markers = match_keywords(head, DOMAIN_MARKERS)
    if cities and markers:
        return True, f"city+context:{cities[0]}", cities[:1]

    if match_keywords(head, ["наукоград", "наукограды", "город науки", "наукоёмк", "наукоемк"]):
        return True, "domain_marker", cities[:1]

    return False, None, []


def classify_rosatom(title, text):
    blob = text_blob(title, text)
    hits = match_keywords(blob, ROSATOM_KEYWORDS)
    return len(hits) >= 2, hits


def classify_international(title, text):
    blob = text_blob(title, text)
    hits = match_keywords(blob, INTL_KEYWORDS)
    return len(hits) >= 2, hits


def infer_metadata(url, title, text, domain):
    blob = text_blob(title, text)
    url_l = (url or "").lower()
    title_l = (title or "").lower()

    voice_type = "media"
    if any(x in url_l for x in ["minobrnauki", "government.ru", "kremlin.ru"]):
        voice_type = "official"
    elif any(x in title_l for x in ["интервью", "беседа", "разговор с"]):
        voice_type = "interview"
    elif any(x in title_l for x in ["выступлен", "доклад", "заявил", "сообщил"]):
        voice_type = "speech"
    elif any(x in url_l for x in ["trends.rbc", "vedomosti", "kommersant", "expert.ru"]):
        voice_type = "analytics"

    text_type = "news"
    if voice_type == "interview":
        text_type = "interview"
    elif voice_type in ("official", "speech"):
        text_type = "speech"
    elif voice_type == "analytics":
        text_type = "analytics"

    naukograd_city = None
    if domain == "naukogrady":
        cities = match_keywords(blob, NAUKOGRAD_CITIES)
        naukograd_city = cities[0].title() if cities else None

    actor = None
    for pattern in [
        r"(Фальков|Мишустин|Путин|Кириенко|Лихачёв|Лихачев|Нуждин|Салихов)",
        r"глав[аы] РАН\s+([А-ЯЁ][а-яё]+(?:\s+[А-ЯЁ][а-яё]+)?)",
    ]:
        m = re.search(pattern, title or "") or re.search(pattern, text[:500] if text else "")
        if m:
            actor = m.group(1) if m.lastindex else m.group(0)
            break

    return {
        "domain": domain,
        "voice_type": voice_type,
        "text_type": text_type,
        "actor": actor,
        "naukograd_city": naukograd_city,
    }


def discover_rbc_article_urls(pages):
    pattern = re.compile(r"/trends/(?:education|science|social)/[a-f0-9]{24}")
    found = set()
    for page_url in pages:
        try:
            resp = requests.get(page_url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
            resp.raise_for_status()
            soup = BeautifulSoup(resp.text, "lxml")
            for a in soup.find_all("a", href=True):
                href = a["href"].split("?")[0]
                if pattern.search(href):
                    from urllib.parse import urljoin
                    found.add(urljoin("https://trends.rbc.ru", href))
        except Exception as e:
            print(f"[RBC DISCOVER ERROR] {page_url} -> {e}")
    return sorted(found)


def discover_google_news_urls(queries, max_per_query=25):
    from urllib.parse import quote
    found = []
    seen = set()
    for query in queries:
        rss_url = (
            "https://news.google.com/rss/search?q="
            f"{quote(query)}&hl=ru&gl=RU&ceid=RU:ru"
        )
        try:
            feed = parse_rss(rss_url)
            for entry in feed.entries[:max_per_query]:
                url = resolve_url(entry.link)
                if url in seen:
                    continue
                seen.add(url)
                found.append({
                    "url": url,
                    "title_hint": entry.get("title", ""),
                    "date": entry.get("published", ""),
                    "source": (
                        entry["source"].get("title", "")
                        if isinstance(entry.get("source"), dict)
                        else str(entry.get("source") or "")
                    ),
                    "query": query,
                })
        except Exception as e:
            print(f"[GNEWS ERROR] {query} -> {e}")
    return found


def discover_rss_urls(feeds, keywords, max_per_feed=40):
    found = []
    seen = set()
    for rss_url, label in feeds:
        try:
            feed = parse_rss(rss_url)
            per_feed = 0
            for entry in feed.entries:
                blob = text_blob(
                    entry.get("title", ""),
                    entry.get("summary", "") + " " + entry.get("link", ""),
                )
                if not match_keywords(blob, keywords):
                    continue
                url = entry.link
                if url in seen:
                    continue
                seen.add(url)
                found.append({
                    "url": url,
                    "title_hint": entry.get("title", ""),
                    "date": entry.get("published", ""),
                    "source": feed.feed.get("title", label),
                })
                per_feed += 1
                if per_feed >= max_per_feed:
                    break
        except Exception as e:
            print(f"[RSS DISCOVER ERROR] {rss_url} -> {e}")
    return found


def make_document(url, parsed, meta, feed_meta=None):
    text = parsed["text"]
    entities = extract_entities(text)
    keywords = extract_keywords(text)
    feed_meta = feed_meta or {}
    doc = {
        "doc_id": str(uuid.uuid4()),
        "source": feed_meta.get("source", ""),
        "url": url,
        "date": feed_meta.get("date", ""),
        "title": parsed["title"],
        "text": text,
        "authors": parsed["authors"],
        "top_image": parsed["top_image"],
        "entities": entities,
        "keywords": keywords,
        **meta,
    }
    if feed_meta.get("query"):
        doc["discovery_query"] = feed_meta["query"]
    if feed_meta.get("relevance_reason"):
        doc["relevance_reason"] = feed_meta["relevance_reason"]
    return doc


def collect_urls(url_items, domain, classifier=None, source_label=""):
    import time
    documents = []
    seen = set()
    stats = Counter()

    for item in tqdm(url_items, desc=f"collect:{domain}"):
        url = item if isinstance(item, str) else item["url"]
        if url in seen:
            continue
        seen.add(url)

        resolved = resolve_url(url)
        parsed = parse_article(resolved)
        time.sleep(SLEEP_BETWEEN_REQUESTS)

        if not parsed or len(parsed["text"]) < MIN_TEXT_LEN:
            stats["skipped_short"] += 1
            continue

        title, text = parsed["title"], parsed["text"]
        relevance_reason = None

        if classifier:
            ok, *extra = classifier(title, text)
            if not ok:
                stats["skipped_irrelevant"] += 1
                continue
            relevance_reason = extra[0] if extra else None

        meta = infer_metadata(resolved, title, text, domain)
        feed_meta = item if isinstance(item, dict) else {}
        if relevance_reason:
            feed_meta = {**feed_meta, "relevance_reason": relevance_reason}
        if source_label and not feed_meta.get("source"):
            feed_meta["source"] = source_label

        documents.append(make_document(resolved, parsed, meta, feed_meta))
        stats["saved"] += 1

    print(f"  [{domain}] saved={stats['saved']} short={stats['skipped_short']} irrelevant={stats['skipped_irrelevant']}")
    return documents


def save_corpus(documents, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(documents, f, ensure_ascii=False, indent=2)
    print(f"Saved {len(documents)} docs -> {path}")


def load_or_migrate_general():
    legacy = os.path.join(RAW_DIR, "documents.json")
    target = os.path.join(RAW_GENERAL, "documents.json")
    if os.path.exists(legacy) and not os.path.exists(target):
        import shutil
        shutil.copy(legacy, target)
        print(f"Migrated legacy corpus -> {target}")
    if os.path.exists(target):
        with open(target, encoding="utf-8") as f:
            return json.load(f)
    return []


def dedupe_documents(docs):
    by_url = {}
    for d in docs:
        url = d.get("url")
        if url and (url not in by_url or len(d.get("text", "")) > len(by_url[url].get("text", ""))):
            by_url[url] = d
    return list(by_url.values())


def build_graph_sentences(docs):
    G = nx.Graph()
    for doc in docs:
        doc_entities = [e["text"].lower() for e in doc.get("entities", [])]
        kws = [k.lower() for k in doc.get("keywords", [])[:10]]
        for sent in split_sentences(doc["text"]):
            sent_l = sent.lower()
            nodes = list({
                e for e in doc_entities if e in sent_l
            } | {
                k for k in kws if k in sent_l
            })
            if len(nodes) < 2:
                continue
            for a, b in combinations(nodes, 2):
                if a == b:
                    continue
                if G.has_edge(a, b):
                    G[a][b]["weight"] += 1
                else:
                    G.add_edge(a, b, weight=1)
    return G


def apply_pmi(G):
    total_edges = sum(data["weight"] for _, _, data in G.edges(data=True))
    if total_edges == 0:
        return G
    node_freq = Counter()
    for u, v, data in G.edges(data=True):
        node_freq[u] += data["weight"]
        node_freq[v] += data["weight"]
    for u, v, data in G.edges(data=True):
        pxy = data["weight"] / total_edges
        px = node_freq[u] / total_edges
        py = node_freq[v] / total_edges
        G[u][v]["pmi"] = round(math.log2(pxy / (px * py)), 4)
    return G


def build_graph_filtered(docs, top_nodes=GRAPH_TOP_NODES, min_weight=GRAPH_MIN_WEIGHT, min_pmi=GRAPH_MIN_PMI):
    G = build_graph_sentences(docs)
    if G.number_of_edges() == 0:
        return G
    deg = Counter()
    for u, v, data in G.edges(data=True):
        deg[u] += data["weight"]
        deg[v] += data["weight"]
    top = {n for n, _ in deg.most_common(top_nodes)}
    G2 = nx.Graph()
    for u, v, data in G.edges(data=True):
        if u not in top or v not in top:
            continue
        if data["weight"] < min_weight:
            continue
        G2.add_edge(u, v, weight=data["weight"])
    G2 = apply_pmi(G2)
    to_remove = [(u, v) for u, v, d in G2.edges(data=True) if d.get("pmi", 0) < min_pmi]
    G2.remove_edges_from(to_remove)
    G2.remove_nodes_from(list(nx.isolates(G2)))
    return G2


def save_graph_outputs(G, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    nx.write_graphml(G, os.path.join(out_dir, "discourse_graph.graphml"))
    edges = [{"source": u, "target": v, "weight": d["weight"], "pmi": d.get("pmi", 0)} for u, v, d in G.edges(data=True)]
    pd.DataFrame(edges).to_csv(os.path.join(out_dir, "edges.csv"), index=False)
    pd.DataFrame([{"id": n} for n in G.nodes()]).to_csv(os.path.join(out_dir, "nodes.csv"), index=False)
    print(f"Graph: nodes={G.number_of_nodes()} edges={G.number_of_edges()} -> {out_dir}")


def save_graphrag(docs, path, domain=None):
    rag_docs = []
    for doc in docs:
        meta = {
            "title": doc["title"],
            "source": doc.get("source", ""),
            "url": doc["url"],
            "date": doc.get("date", ""),
            "domain": doc.get("domain", domain),
            "voice_type": doc.get("voice_type"),
            "text_type": doc.get("text_type"),
            "actor": doc.get("actor"),
            "naukograd_city": doc.get("naukograd_city"),
        }
        rag_docs.append({"id": doc["doc_id"], "text": doc["text"], "metadata": meta})
    with open(path, "w", encoding="utf-8") as f:
        json.dump(rag_docs, f, ensure_ascii=False, indent=2)
    print(f"GraphRAG docs ({len(rag_docs)}) -> {path}")

In [ ]:
# =========================
# СБОР КОРПУСОВ
# =========================

corpora = {}

if COLLECT.get("general_science"):
    print("\n=== GENERAL SCIENCE (RSS) ===")
    general_urls = [
        {"url": e.link, "date": e.get("published", ""), "source": "rss"}
        for rss_url in RSS_FEEDS_GENERAL
        for e in parse_rss(rss_url).entries
    ]
    general_docs = collect_urls(general_urls, domain="general_science", classifier=None)
    general_docs = dedupe_documents(general_docs)
    path = os.path.join(RAW_GENERAL, "documents.json")
    save_corpus(general_docs, path)
    corpora["general_science"] = general_docs
else:
    corpora["general_science"] = load_or_migrate_general()
    print(f"general_science: {len(corpora['general_science'])} docs (existing)")

if COLLECT.get("naukogrady"):
    print("\n=== НАУКОГРАДЫ ===")
    naukograd_urls = []

    # 1) Сиды + RBC Trends
    naukograd_urls.extend(SEED_URLS_NAUKOGRADY)
    rbc_urls = discover_rbc_article_urls(RBC_EDUCATION_PAGES)
    print(f"RBC Trends articles discovered: {len(rbc_urls)}")
    naukograd_urls.extend(rbc_urls)

    # 2) Google News
    gnews = discover_google_news_urls(GOOGLE_NEWS_NAUKOGRADY, max_per_query=25)
    print(f"Google News candidates: {len(gnews)}")
    naukograd_urls.extend(gnews)

    # 3) Поиск по каждому городу
    city_queries = [
        f"{name} наукоград" for name in [
            "Долгопрудный", "Дубна", "Обнинск", "Королёв", "Фрязино", "Пущино",
            "Жуковский", "Реутов", "Троицк", "Черноголовка", "Сколково", "Иннополис",
        ]
    ]
    city_gnews = discover_google_news_urls(city_queries, max_per_query=15)
    print(f"City Google News candidates: {len(city_gnews)}")
    naukograd_urls.extend(city_gnews)

    # 4) RSS с предфильтром по заголовку
    rss_hits = discover_rss_urls(RSS_FILTERED, RSS_MATCH_KEYWORDS, max_per_feed=40)
    print(f"Filtered RSS candidates: {len(rss_hits)}")
    naukograd_urls.extend(rss_hits)

    # Дедуп URL-кандидатов
    def _url_key(item):
        return item if isinstance(item, str) else item["url"]

    seen_urls = set()
    seed_items = [{"url": u, "source": "seed"} for u in SEED_URLS_NAUKOGRADY]
    other_items = []
    for item in naukograd_urls:
        u = _url_key(item)
        if u in seen_urls:
            continue
        seen_urls.add(u)
        if u in SEED_URLS_NAUKOGRADY:
            continue
        other_items.append(item if isinstance(item, dict) else {"url": item})

    naukograd_docs = []
    naukograd_docs.extend(collect_urls(seed_items, domain="naukogrady", classifier=None))
    naukograd_docs.extend(collect_urls(other_items, domain="naukogrady", classifier=classify_naukograd))

    naukograd_docs = dedupe_documents(naukograd_docs)
    path = os.path.join(RAW_NAUKOGRADY, "documents.json")
    save_corpus(naukograd_docs, path)
    save_graphrag(naukograd_docs, os.path.join(PROCESSED_DIR, "naukogrady_graphrag_docs.json"), domain="naukogrady")
    corpora["naukogrady"] = naukograd_docs




general_science: 0 docs (existing)

=== НАУКОГРАДЫ ===
RBC Trends articles discovered: 21
Google News candidates: 189
City Google News candidates: 172
Filtered RSS candidates: 35


collect:naukogrady:  75%|███████▌  | 3/4 [00:02<00:00,  1.16it/s]

[ARTICLE ERROR] https://trends.rbc.ru/trends/education/6440cd219a7947834e9e39d0 -> Article `download()` failed with 406 Client Error: Not Acceptable for url: https://trends.rbc.ru/trends/education/6440cd219a7947834e9e39d0 on URL https://trends.rbc.ru/trends/education/6440cd219a7947834e9e39d0


collect:naukogrady: 100%|██████████| 4/4 [00:02<00:00,  1.34it/s]


  [naukogrady] saved=3 short=1 irrelevant=0


...

collect:naukogrady:  73%|███████▎  | 260/357 [04:09<01:59,  1.23s/it]

[ARTICLE ERROR] https://golosinfo.org/articles/146331 -> Article `download()` failed with HTTPSConnectionPool(host='golosinfo.org', port=443): Read timed out. (read timeout=7) on URL https://golosinfo.org/articles/146331


collect:naukogrady: 100%|██████████| 357/357 [05:44<00:00,  1.04it/s]


  [naukogrady] saved=218 short=74 irrelevant=65
Saved 221 docs -> data/raw/naukogrady/documents.json
GraphRAG docs (221) -> data/processed/naukogrady_graphrag_docs.json

=== РОСАТОМ (контроль) ===


KeyboardInterrupt: 

In [12]:
# Сводка
print("\n=== SUMMARY ===")
for name, docs in corpora.items():
    voices = Counter(d.get("voice_type") for d in docs)
    actors = sum(1 for d in docs if d.get("actor"))
    cities = Counter(d.get("naukograd_city") for d in docs if d.get("naukograd_city"))
    print(f"{name}: {len(docs)} docs | voices={dict(voices)} | with_actor={actors}")
    if name == "naukogrady" and cities:
        print(f"  top cities: {cities.most_common(8)}")


=== SUMMARY ===
general_science: 0 docs | voices={} | with_actor=0
naukogrady: 221 docs | voices={'analytics': 12, 'media': 205, 'official': 2, 'interview': 2} | with_actor=27
  top cities: [('Сколково', 29), ('Дубна', 25), ('Долгопрудный', 16), ('Реутов', 16), ('Иннополис', 14), ('Обнинск', 13), ('Фрязино', 10), ('Пущино', 9)]


In [13]:
# =========================
# ГРАФ (наукограды, sentence-level + фильтры)
# =========================

naukograd_path = os.path.join(RAW_NAUKOGRADY, "documents.json")

if os.path.exists(naukograd_path):
    with open(naukograd_path, encoding="utf-8") as f:
        naukograd_docs = json.load(f)

    print(f"Building filtered graph from {len(naukograd_docs)} documents...")
    G = build_graph_filtered(naukograd_docs)
    save_graph_outputs(G, GRAPH_NAUKOGRADY)

    # объединённый индекс для экспериментов
    all_docs = []
    for subdir in ["naukogrady", "rosatom", "international", "general_science"]:
        p = os.path.join(RAW_DIR, subdir, "documents.json")
        if os.path.exists(p):
            with open(p, encoding="utf-8") as f:
                all_docs.extend(json.load(f))

    all_docs = dedupe_documents(all_docs)
    save_graphrag(all_docs, os.path.join(PROCESSED_DIR, "graphrag_docs_all.json"))
    print(f"\nAll corpora combined: {len(all_docs)} unique documents")
else:
    print("Нет data/raw/naukogrady/documents.json — сначала запустите ячейку сбора")

print("\nDONE")

Building filtered graph from 221 documents...
Graph: nodes=253 edges=317 -> data/graphs/naukogrady
GraphRAG docs (221) -> data/processed/graphrag_docs_all.json

All corpora combined: 221 unique documents

DONE
